In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset

# 1. 读取数据
train_df = pd.read_csv("train.csv")
btest_df = pd.read_csv("Btest.csv")


# 2. 划分训练集 / 验证集（比如 9:1）
train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["label"]
)

# 3. 转成 HuggingFace Dataset（方便和 transformers 对接）
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
btest_dataset = Dataset.from_pandas(btest_df.reset_index(drop=True))

print(train_dataset[0])


{'id': 1558, 'text': '摸奶节是中国云南双柏县鄂家镇彝族传统文化的庆典就是农历的7月14日、15日与16日这三天，包括外来的游人,如果在街上遇见喜欢的女子，都可以摸一摸女子的胸部。姑娘们表面躲躲闪闪，但决无责怪之意因为这是他们这个地区的百姓延续了1000多年的风俗。小伙子以摸到奶为吉祥，姑娘们以被摸奶为幸运和祝福!', 'label': 1}


In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 使用中文 DeBERTa-v2 模型（Erlangshen 系列）
model_name = "IDEA-CCNL/Erlangshen-DeBERTa-v2-97M-Chinese"

# 自动加载对应的分词器（可能是 SentencePiece / WordPiece，AutoTokenizer 会自动识别）
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 自动加载用于分类任务的模型，并指定 num_labels=2（二分类）
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at IDEA-CCNL/Erlangshen-DeBERTa-v2-97M-Chinese and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
max_length = 256  # 注意：必须与训练时一致（你 BERT 用 128 就保持 128）

def preprocess_function(examples):
    result = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length
    )
    return result

# 用新的 tokenizer 对三个数据集重新编码
train_encoded = train_dataset.map(preprocess_function, batched=True)
val_encoded = val_dataset.map(preprocess_function, batched=True)
btest_encoded = btest_dataset.map(preprocess_function, batched=True)



Map:   0%|          | 0/3305 [00:00<?, ? examples/s]

Map:   0%|          | 0/368 [00:00<?, ? examples/s]

Map:   0%|          | 0/1225 [00:00<?, ? examples/s]

In [12]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
# 评估指标函数（沿用之前的）
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    # AUC 使用正类的连续得分
    probs = logits[:, 1]
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float("nan")
    return {"accuracy": acc, "f1": f1, "auc": auc}

In [13]:
from transformers import TrainingArguments, Trainer

# 为 DeBERTa 单独配置训练参数
training_args = TrainingArguments(
    output_dir="./deberta_fake_news",     # 换新的输出目录
    num_train_epochs=4,                   # 先保持和 BERT 一样，方便对比
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    weight_decay=0.01,
    logging_dir="./deberta_fake_news_logs",
    logging_steps=50
)

# 构建 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_encoded,
    eval_dataset=val_encoded,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)



C:\Users\fall\AppData\Local\Temp\ipykernel_34612\1691060913.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [14]:
# 训练 DeBERTa 模型
trainer.train()

# 在验证集上评估
metrics = trainer.evaluate()
print(metrics)

Step,Training Loss
50,0.528300
100,0.408800
150,0.328000
200,0.274300
250,0.162600
300,0.192400
350,0.151300
400,0.125400
450,0.081200
500,0.063000


{'eval_loss': 0.2656286954879761, 'eval_accuracy': 0.9402173913043478, 'eval_f1': 0.9367816091954023, 'eval_auc': 0.987942884227989, 'eval_runtime': 12.0578, 'eval_samples_per_second': 30.52, 'eval_steps_per_second': 0.995, 'epoch': 4.0}


In [15]:
import torch
import numpy as np

predictions = trainer.predict(btest_encoded)

logits = predictions.predictions          # shape: [N, 2]
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()

prob_fake = probs[:, 1]                   # 正类（label=1=假新闻）概率

# 生成提交文件
submission_b = pd.DataFrame({
    "id": btest_df["id"],
    "prob": prob_fake
})

submission_b.to_csv("deberta_submission_b.csv", index=False, encoding="utf-8")
print("Saved: deberta_submission_b.csv")



Saved: deberta_submission_b.csv
